In [1]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os

# Define the directories for AZ and WY
base_dir = r'H:\My Drive\DroughtSurfing'
# locations = ['WY', 'AZ', 'UT']
locations = ['UT']

# Define file paths for the exported GeoTIFFs within each directory
ndvi_file_names = [
    'smoothed_ndvi.tif'
]

In [2]:
def calculate_global_quantiles(ndvi_array, lower_percentile=2.5, upper_percentile=97.5):
    """
    Calculate the global lower and upper quantiles for the NDVI values.
    
    Args:
    - ndvi_array: The numpy array of NDVI values.
    - lower_percentile: The lower percentile to compute (default 2.5).
    - upper_percentile: The upper percentile to compute (default 97.5).
    
    Returns:
    - lower_quantile: The value at the lower percentile.
    - upper_quantile: The value at the upper percentile.
    """
    lower_quantile = np.percentile(ndvi_array, lower_percentile)
    upper_quantile = np.percentile(ndvi_array, upper_percentile)
    
    return lower_quantile, upper_quantile

def scale_ndvi(ndvi_array, lower_quantile, upper_quantile):
    """
    Scale NDVI values between 0 and 1 based on the global lower and upper quantiles.
    
    Args:
    - ndvi_array: The numpy array of NDVI values.
    - lower_quantile: The value at the lower percentile (e.g., 0.025 quantile).
    - upper_quantile: The value at the upper percentile (e.g., 0.975 quantile).
    
    Returns:
    - scaled_ndvi: The NDVI values scaled between 0 and 1.
    """
    # Scale NDVI using the formula: (NDVI - lower_quantile) / (upper_quantile - lower_quantile)
    scaled_ndvi = (ndvi_array - lower_quantile) / (upper_quantile - lower_quantile)
    
    # Ensure values are constrained between 0 and 1
    scaled_ndvi = np.clip(scaled_ndvi, 0, 1)
    
    return scaled_ndvi

def process_ndvi_files(base_dir, locations, ndvi_file_names):
    """
    Process the NDVI files by scaling them without saving the result.
    
    Args:
    - base_dir: The base directory containing NDVI files.
    - locations: List of location directories.
    - ndvi_file_names: List of NDVI file names to process.
    
    Returns:
    - scaled_ndvi_dict: A dictionary with scaled NDVI arrays keyed by location and file name.
    """
    scaled_ndvi_dict = {}
    
    for location in locations:
        location_dir = os.path.join(base_dir, location)
        
        for ndvi_file in ndvi_file_names:
            file_path = os.path.join(location_dir, ndvi_file)
            if not os.path.exists(file_path):
                print(f"File {file_path} does not exist.")
                continue
            
            # Open the NDVI GeoTIFF
            with rasterio.open(file_path) as src:
                ndvi_array = src.read(1)  # Read the first band (assuming it's NDVI)
                
                # Calculate the global quantiles
                lower_quantile, upper_quantile = calculate_global_quantiles(ndvi_array)
                print(f"Lower Quantile (2.5%): {lower_quantile}")
                print(f"Upper Quantile (97.5%): {upper_quantile}")
                
                # Scale the NDVI values based on the quantiles
                scaled_ndvi = scale_ndvi(ndvi_array, lower_quantile, upper_quantile)
                
                # Store the result in the dictionary
                scaled_ndvi_dict[(location, ndvi_file)] = scaled_ndvi
    
    return scaled_ndvi_dict

# Process the NDVI files and get the scaled arrays
scaled_ndvi_dict = process_ndvi_files(base_dir, locations, ndvi_file_names)



Lower Quantile (2.5%): nan
Upper Quantile (97.5%): nan
Scaled NDVI for UT - smoothed_ndvi.tif:
[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


In [ ]:
export_collection_as_stack(scaled_ndvi_collection.select('NDVI_scaled'), 'scaled_ndvi', 'DroughtSurfing') ## only select the scaled bands
